In [1]:
import torch
import snntorch as snn
from matplotlib import pyplot as plt

from sklearn import datasets
from sklearn.model_selection import train_test_split
from torch import nn

import sys
sys.path.append(f"../model_compiler")
from neuron_mapper import Neuron_Mapper, Neuron_LIF

In [2]:
# --- Configurations ---
CUT_TRAINING_SHORT = False
TRAINING_CUTOFF = 1

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float32

input_nodes = 4       # Sepal length, sepal width, petal length, petal width
hidden_neurons = 16  # Arbitrary hidden layer size
output_neurons = 3    # Setosa, Virginica, Versicolor

threshold = 1      # LIF neuron threshold
beta = Neuron_LIF.DECAY_MODE_LIF_2       # LIF neuron decay rate
dt_steps = 50         # Time steps (rate coding duration)
epochs = 1000         # Training epochs

In [3]:
# --- Load and Prepare Data ---
iris = datasets.load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=0)

# Normalize and convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=dtype)
X_test_tensor = torch.tensor(X_test, dtype=dtype)

X_train_tensor /= X_train_tensor.max()
X_test_tensor /= X_test_tensor.max()

In [4]:
def rate_encode_poisson(inputs, num_steps):
    """
    inputs: [batch, features] — normalized to [0, 1]
    num_steps: number of time steps
    returns: [num_steps, batch, features] — binary spikes
    """
    # Normalize input to [0, 1]
    max_vals = inputs.max(dim=0, keepdim=True).values
    inputs_norm = inputs / max_vals.clamp(min=1e-6)

    # Generate spikes with probability = input value
    spike_prob = inputs_norm.unsqueeze(0).expand(num_steps, -1, -1)
    spikes = torch.rand_like(spike_prob) < spike_prob
    return spikes.float()

In [5]:
# Poisson encode to spike trains over time
train_encoded = rate_encode_poisson(X_train_tensor, dt_steps).to(device)
test_encoded = rate_encode_poisson(X_test_tensor, dt_steps).to(device)

In [6]:
# --- Define Spiking Model ---
class SpikingIrisClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(input_nodes, hidden_neurons, bias=False)   # No bias here
        self.lif1 = snn.Leaky(beta=beta, threshold=threshold)
        self.transfer1 = nn.Linear(hidden_neurons, output_neurons, bias=False)  # No bias here
        self.output = snn.Leaky(beta=beta, threshold=threshold)

    def forward(self, x_spike_seq):  # x_spike_seq: [time, batch, input]
        mem1 = self.lif1.init_leaky()
        mem2 = self.output.init_leaky()

        spike_record = []

        for t in range(dt_steps):
            x_t = x_spike_seq[t]
            cur1 = self.input(x_t)
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.transfer1(spk1)
            spk2, mem2 = self.output(cur2, mem2)

            spike_record.append(spk2)

        # Accumulate spikes across time
        spike_sum = torch.stack(spike_record, dim=0).sum(dim=0)  # [batch, output_neurons]
        return spike_sum


In [7]:
# --- Training Function ---
def train_classifier(classifier, loss_function, optimizer, inputs, targets):
    print("Training started...\n")
    loss_history = []

    targets = torch.LongTensor(targets).to(device)

    for epoch in range(epochs):
        classifier.train()
        optimizer.zero_grad()

        outputs = classifier(inputs)
        loss = loss_function(outputs, targets)

        loss.backward()
        optimizer.step()

        loss_value = loss.item()
        loss_history.append(loss_value)

        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss_value:.4f}")

        if CUT_TRAINING_SHORT and loss_value < TRAINING_CUTOFF:
            break

    return loss_history

In [8]:
def test_classifier(classifier, inputs, targets):
    print("\nTesting started...\n")
    classifier.eval()

    with torch.no_grad():
        outputs = classifier(inputs)
        _, predicted = torch.max(outputs, 1)
        actual = torch.LongTensor(targets).to(device)

        correct = (predicted == actual).sum().item()
        total = actual.size(0)

    return correct, total

In [9]:
# --- Run Everything ---
classifier = SpikingIrisClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=5e-4)

train_loss = train_classifier(classifier, loss_function, optimizer, train_encoded, y_train)
correct, total = test_classifier(classifier, test_encoded, y_test)

# --- Report Accuracy ---
print("------")
print(f"Test Accuracy: {100 * correct / total:.2f}% ({correct}/{total})")
print("------")

Training started...

Epoch 1/1000 - Loss: 1.0986
Epoch 2/1000 - Loss: 1.0986
Epoch 3/1000 - Loss: 1.0986
Epoch 4/1000 - Loss: 1.0986
Epoch 5/1000 - Loss: 1.0986
Epoch 6/1000 - Loss: 1.0986
Epoch 7/1000 - Loss: 1.0986
Epoch 8/1000 - Loss: 1.0986
Epoch 9/1000 - Loss: 1.0986
Epoch 10/1000 - Loss: 1.0986
Epoch 11/1000 - Loss: 1.0986
Epoch 12/1000 - Loss: 1.0986
Epoch 13/1000 - Loss: 1.0986
Epoch 14/1000 - Loss: 1.0986
Epoch 15/1000 - Loss: 1.0986
Epoch 16/1000 - Loss: 1.0986
Epoch 17/1000 - Loss: 1.0986
Epoch 18/1000 - Loss: 1.0986
Epoch 19/1000 - Loss: 1.0986
Epoch 20/1000 - Loss: 1.0986
Epoch 21/1000 - Loss: 1.0986
Epoch 22/1000 - Loss: 1.0986
Epoch 23/1000 - Loss: 1.0986
Epoch 24/1000 - Loss: 1.0986
Epoch 25/1000 - Loss: 1.0986
Epoch 26/1000 - Loss: 1.0986
Epoch 27/1000 - Loss: 1.0986
Epoch 28/1000 - Loss: 1.0986
Epoch 29/1000 - Loss: 1.0986
Epoch 30/1000 - Loss: 1.0986
Epoch 31/1000 - Loss: 1.0986
Epoch 32/1000 - Loss: 1.0986
Epoch 33/1000 - Loss: 1.0986
Epoch 34/1000 - Loss: 1.0986
Ep

In [10]:
# mapping model
neuron_layers = {
    "Input" : {
        "decay_mode": beta,
        "threshold": threshold,
        "neurons": input_nodes,
        "reset_mode": Neuron_LIF.RESET_MODE_ZERO,
        "is_virtual": True
    },
    "Layer1": {
        "decay_mode": beta,
        "threshold": threshold,
        "neurons": hidden_neurons,
        "reset_mode": Neuron_LIF.RESET_MODE_ZERO
    },
    "Layer2": {
        "decay_mode": beta,
        "threshold": threshold,
        "neurons": output_neurons,
        "reset_mode": Neuron_LIF.RESET_MODE_ZERO
    }
}

neuron_weights = {
    "Input-Layer1": {
        "weights": classifier.input.weight.detach().cpu().numpy().tolist()
    },
    "Layer1-Layer2": {
        "weights": classifier.transfer1.weight.detach().cpu().numpy().tolist()
    }
}

print(neuron_weights)

{'Input-Layer1': {'weights': [[-0.0806928351521492, 0.10326328128576279, 0.2884129583835602, 0.4033578932285309], [-0.0029286339413374662, 0.12267262488603592, -0.5438386797904968, -0.45823097229003906], [0.23682497441768646, -0.03119024820625782, -0.0767526775598526, -0.24804745614528656], [-0.04881902411580086, -0.4963200092315674, -0.0462460070848465, -0.07301846891641617], [0.23568393290042877, 0.4863121509552002, -0.21496935188770294, 0.43737342953681946], [0.7023902535438538, 0.5594679117202759, 0.1779319941997528, -0.21074701845645905], [0.38122689723968506, 0.7751592993736267, -0.44531428813934326, -0.2477862685918808], [0.0653865858912468, -0.19304007291793823, 0.4915427565574646, 0.42916402220726013], [0.05835900455713272, 0.21303874254226685, -0.633152186870575, -0.20748141407966614], [0.5707487463951111, -0.24749141931533813, 0.6774138808250427, 0.4418972432613373], [-0.08308713138103485, -0.4487847685813904, 0.21533025801181793, 0.11277367174625397], [0.01883002743124962, 

In [11]:
neuron_mapper = Neuron_Mapper(neuron_layers, neuron_weights)
neuron_mapper.map()

# write to file
output_file = "neuron_mapping.txt"
with open(output_file, 'w') as f:
    for line in neuron_mapper.generate_init():
        f.write(" ".join(line) + "\n")

new_cluster_group 128
made cluster group 128
made cluster 0
weight_bucket_check 4 16
clusters [32]
0 0 True
1 1 True
2 2 True
3 3 True
clusters [0]
4 4 True
5 5 True
6 6 True
7 7 True
8 8 True
9 9 True
10 10 True
11 11 True
12 12 True
13 13 True
14 14 True
15 15 True
16 16 True
17 17 True
18 18 True
19 19 True
00010
00100
00001
00000
00000
00000010
00000001
00000000
00000000
00000000
00000000
00000000
00000000
00000000


In [12]:
neuron_mapper.log_mapping()

Virtual Cluster ID: 32
Neuron ID: 0, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 1, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 2, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 3, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Next Neuron ID: 4
Neurons Full: False
Layer: Layer1, Clusters: [0]
Cluster ID: 0
Neurons: 16
Neuron ID: 0, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 1, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 2, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 3, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 4, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 5, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 6, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 7, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 8, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 9, Cluster ID: 0, Decay Mode: 0.5, Threshold: 1
Neuron ID: 10, Cluster ID: 0, Decay Mode: 0.5,

In [13]:
print(X_test.shape)
print(test_encoded.shape)

(30, 4)
torch.Size([50, 30, 4])


In [14]:
with open("test_values.txt", 'w') as f:
    for sample in range(test_encoded.shape[1]):  # input samples
        for t in range(test_encoded.shape[0]):  # time steps
            for feature in range(test_encoded.shape[2]):  # 4 features
                value = test_encoded[t][sample][feature]
                if value > 0:
                    cluster_id = 32  # 6 bits
                    neuron_id = feature  # 5 bits
                    packet = (cluster_id << 5) | neuron_id  # 6-bit cluster + 5-bit neuron
                    f.write(f"{packet:03X}\n")
                else:
                    f.write("FFF\n")

In [15]:
print("outputs")
print(y_test)

outputs
[2 1 0 2 0 2 0 1 1 1 2 1 1 1 1 0 1 1 0 0 2 1 0 0 2 0 0 1 1 0]


In [16]:
with open("test_labels.txt", 'w') as f:
    for label in y_test:
        f.write(f"{label}\n")

In [17]:
def compress_weights_with_dict(weights, precision=5):
    """Compresses a weight matrix using dictionary encoding."""
    # Round and flatten all weights
    rounded_weights = [[round(w, precision) for w in row] for row in weights]
    unique_values = sorted(set(w for row in rounded_weights for w in row))
    value_to_index = {v: i for i, v in enumerate(unique_values)}

    # Compress weights to indices
    index_map = [[value_to_index[w] for w in row] for row in rounded_weights]
    print(f"Unique values: {len(unique_values)}")
    return unique_values, index_map

def decompress_indices_to_tensor(index_map, value_dict):
    """Decompresses index map to a tensor using the value dictionary."""
    decompressed = [[value_dict[i] for i in row] for row in index_map]
    return torch.tensor(decompressed, dtype=torch.float32)

class CompressedSpikingIrisClassifier(nn.Module):
    def __init__(self, dict_input_layer, map_input_layer, dict_hidden_layer, map_hidden_layer):
        super().__init__()

        # Create layers with correct shapes
        self.input = nn.Linear(input_nodes, hidden_neurons, bias=False)
        self.transfer1 = nn.Linear(hidden_neurons, output_neurons, bias=False)
        self.lif1 = snn.Leaky(beta=beta, threshold=threshold)
        self.output = snn.Leaky(beta=beta, threshold=threshold)

        # Decompress and set weights
        input_weight_tensor = decompress_indices_to_tensor(map_input_layer, dict_input_layer)
        hidden_weight_tensor = decompress_indices_to_tensor(map_hidden_layer, dict_hidden_layer)

        self.input.weight = nn.Parameter(input_weight_tensor)
        self.transfer1.weight = nn.Parameter(hidden_weight_tensor)

    def forward(self, x_spike_seq):
        mem1 = self.lif1.init_leaky()
        mem2 = self.output.init_leaky()

        spike_record = []

        for t in range(dt_steps):
            x_t = x_spike_seq[t]
            cur1 = self.input(x_t)
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.transfer1(spk1)
            spk2, mem2 = self.output(cur2, mem2)

            spike_record.append(spk2)

        spike_sum = torch.stack(spike_record, dim=0).sum(dim=0)
        return spike_sum


In [18]:
# --- After training ---
# Step 1: Extract and compress weights
input_w = classifier.input.weight.detach().cpu().numpy().tolist()
hidden_w = classifier.transfer1.weight.detach().cpu().numpy().tolist()

dict_input, map_input = compress_weights_with_dict(input_w, precision=1)
dict_hidden, map_hidden = compress_weights_with_dict(hidden_w, precision=1)

# Step 2: Build compressed model
compressed_model = CompressedSpikingIrisClassifier(
    dict_input_layer=dict_input,
    map_input_layer=map_input,
    dict_hidden_layer=dict_hidden,
    map_hidden_layer=map_hidden
).to(device)

# Step 3: Test compressed model
correct, total = test_classifier(compressed_model, test_encoded, y_test)

print("------")
print(f"Test Accuracy (Compressed Model): {100 * correct / total:.2f}% ({correct}/{total})")
print("------")


Unique values: 15
Unique values: 10

Testing started...

------
Test Accuracy (Compressed Model): 93.33% (28/30)
------
